In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from transformers import AutoTokenizer

In [5]:
with open("../data/rag_text/chunked_docs.json", "r", encoding="utf-8") as f:
    all_posts = json.load(f)

In [6]:
%%capture
# Create documents for RAG
docs = []
for p in all_posts:
    docs.append(Document(page_content=p["content_text"], metadata={
        "author": p["author_name"],
        "topic": p["topic_slug"],
        "created_at": p["created_at"]
    }))

# Text Embedding & Chunking 
embedding_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", use_fast=False)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    embedding_tokenizer,
    chunk_size=500,
    chunk_overlap=50
)
chunked_docs = text_splitter.split_documents(docs)

with open("../data/rag_text/chunked_docs.json", "w", encoding="utf-8") as f:
    json.dump([{"page_content": doc.page_content, "metadata": doc.metadata} for doc in chunked_docs], f, ensure_ascii=False, indent=4)

In [7]:
# # --- Embeddings & Vektorstore aufbauen --- #
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   encode_kwargs={"normalize_embeddings": True})

vectorstore = FAISS.from_documents(chunked_docs, embedding=embeddings)

In [8]:
vectordb_path = "../data/rag_db"
vectorstore.save_local(vectordb_path)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
question = "How do I setup my car best in Forza Horizon 5?"
retrieved_docs = retriever.invoke(question)
retrieved_docs